# market_neural_net — Colab training launcher

Reusable GPU launcher shell per the project README (§2.1: local CPU handles data/backtesting, this notebook handles GPU-hungry training).

Code comes straight from GitHub (https://github.com/yuvidewan/market_neural_net) — no manual zip/upload step needed for that anymore.

**One-time setup, since the repo is private:** a plain `git clone` fails in Colab with `fatal: could not read Username for 'https://github.com'` because there's nothing to authenticate with. Fix, once:
1. GitHub → Settings → Developer settings → Personal access tokens → **Fine-grained tokens** → generate one scoped to just this repo, read-only "Contents" permission.
2. In Colab, click the 🔑 key icon in the left sidebar → Add secret → name it `GITHUB_TOKEN`, paste the token → toggle "Notebook access" on.

Cell 2 below reads that secret automatically (never prints it, and scrubs the token back out of the repo's local git config right after cloning/pulling). If no secret is found it falls back to a plain clone, which only works if you ever make the repo public.

Only the curated **data** needs a one-time Drive upload (it stays out of git on purpose — see `.gitignore`):
- Run `python -m scripts.package_for_colab --skip-data` locally if you ever want the old zip-based code path instead (e.g. offline, or before you've pushed a change) — not needed for normal use now.
- Upload `experiments/colab_data_bundle.zip` (~350MB, produced by the same script) to `My Drive/market_neural_net/colab_data_bundle.zip` — once, and again only after a real re-ingest of the curated dataset.

**Checkpoint/resume exists** (per-epoch, per-fold — `src/train/checkpointing.py`). Design, after a real failure taught us the naive version: `--out-dir` stays on the Colab VM's **local disk** (fast, reliable), and `--mirror-dir` points at Drive as a **best-effort backup** — every real-run cell below passes both. Local-first matters because Drive's FUSE mount doesn't handle interrupted writes reliably; an earlier version that wrote checkpoints straight to Drive could get its resume state corrupted by a write cut off mid-way, which crashed the *next* resume attempt too — the opposite of what checkpointing is for. The current design survives that: local writes are always reliable within a session, and if local disk gets wiped by a full VM disconnect/recycle, the next run auto-hydrates from whatever the Drive mirror last caught up to. Pass `--fresh` on a cell's command to discard existing checkpoints (local and mirrored) and start that run over. The smoke-test cells don't bother with any of this — cheap to lose, not worth it.

## 1. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU assigned — Runtime > Change runtime type > GPU, then re-run this cell.')

## 2. Clone the repo

In [ ]:
REPO_SLUG = 'yuvidewan/market_neural_net'
PLAIN_URL = f'https://github.com/{REPO_SLUG}.git'
PROJECT_DIR = '/content/market_neural_net'

import os
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None

if not token:
    print('No GITHUB_TOKEN Colab secret found -- this only works if the repo is public.')
    print('For a private repo: GitHub > Settings > Developer settings > Personal access')
    print('tokens > Fine-grained tokens > generate one, read-only, scoped to this repo.')
    print('Then in Colab: key icon (left sidebar) > Add secret > name it GITHUB_TOKEN >')
    print('paste the token > enable notebook access > re-run this cell.')

auth_url = f'https://{token}@github.com/{REPO_SLUG}.git' if token else PLAIN_URL

if os.path.exists(PROJECT_DIR):
    %cd $PROJECT_DIR
    !git remote set-url origin $auth_url
    !git pull
    !git remote set-url origin $PLAIN_URL   # scrub the token back out of .git/config
else:
    !git clone $auth_url $PROJECT_DIR
    %cd $PROJECT_DIR
    !git remote set-url origin $PLAIN_URL   # scrub the token back out of .git/config

## 3. Mount Drive and unpack the curated data
Data stays out of git (see `.gitignore`) — this is the one thing that still needs a manual Drive upload.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR = '/content/drive/My Drive/market_neural_net'  # change if you uploaded elsewhere

In [ ]:
import zipfile, os

data_zip = f'{DRIVE_PROJECT_DIR}/colab_data_bundle.zip'
assert os.path.exists(data_zip), (
    f'missing {data_zip} — run `python -m scripts.package_for_colab` locally and upload '
    f'experiments/colab_data_bundle.zip to that Drive path first'
)
with zipfile.ZipFile(data_zip) as zf:
    zf.extractall(PROJECT_DIR)
print('data extracted into', PROJECT_DIR)

## 4. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 5. Sanity checks: tests, then a tiny v2 smoke-test run
Quick pytest pass plus a tiny-scale v2 transformer run — cheap insurance that the code (especially the just-added warmup/clipping fix) actually works before committing to the multi-hour real run below.

In [ ]:
!python -m pytest tests/ -q

In [ ]:
# v2 transformer tiny smoke test -- should also finish in under a minute.
!python -u -m scripts.train_transformer_ssl \
  --n-symbols 5 --seq-len 16 --d-model 16 --n-heads 2 --n-blocks 1 --patch-size 4 --epochs 1 \
  --test-years 2024 --out-dir experiments/transformer_smoketest

## 6. v2 real run: two-axis Transformer (README's "main model")
Mean OOS rank IC > 0.02, stable sign across all 4 walk-forward folds — same bar as M3's already-passing TCN result (0.0252 overall IC).

**`--fresh` is deliberate here**: the first attempt at this run collapsed during training (diagnosed and fixed — see README), but it did complete all 4 folds, so its checkpoint state has them marked "already done" with the bad results cached. Without `--fresh` a re-run would just skip straight to reusing that broken data instead of actually retraining. This flag clears that out, local and mirrored, so this is a genuine fresh start with the fix in place.

This run does far more optimizer steps than the TCN run (one per calendar date rather than one per fixed-size batch), so it's the one checkpoint/resume matters most for — covered via local `--out-dir` + Drive `--mirror-dir`, so a disconnect just means re-running this same cell (drop `--fresh` on any re-run after this one, or it'll wipe progress again).

In [ ]:
!python -u -m scripts.train_transformer_ssl \
  --n-symbols 200 --seq-len 120 --d-model 256 --n-heads 8 --n-blocks 8 --patch-size 16 --epochs 8 \
  --test-years 2022 2023 2024 2025 \
  --out-dir experiments/ssl_quantile_transformer \
  --mirror-dir "{DRIVE_PROJECT_DIR}/experiments/ssl_quantile_transformer" \
  --fresh

## 7. Sync results back to Drive
`--mirror-dir` only backs up checkpoints/resume-state as training goes — the final `report.json` itself is written once, locally, at the very end of a run. Run this cell after the run above completes to get everything, including the final report, onto Drive so you can download it.

In [ ]:
import shutil
dest = f'{DRIVE_PROJECT_DIR}/experiments_from_colab'
shutil.copytree('experiments', dest, dirs_exist_ok=True)
print('synced experiments/ ->', dest)